# 01 · Exploring the Spotify data with SQL

The CSV now lives in a SQLite database (`data/spotify.db`, table `tracks_raw`), loaded by `python -m src.db`.
In this notebook we ask it questions in SQL before doing any machine learning.

**Why SQL first?** Almost every data job stores data in a database, and SQL is how you ask it things.
It's also quick: the database does the counting and averaging for us.

## Setup

`sql()` is a tiny helper: give it a query as text, and it returns the answer as a pandas table so it prints nicely.

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path("..") / "data" / "spotify.db"
conn = sqlite3.connect(DB_PATH)

def sql(query):
    return pd.read_sql(query, conn)

## 1. Look at a few rows: `SELECT` and `LIMIT`

- `SELECT` picks which columns you want (`*` means all of them).
- `FROM` says which table.
- `LIMIT 5` stops after 5 rows. Always use it when peeking at a big table.

In [2]:
sql("""
SELECT track_name, artists, track_genre, popularity, energy, tempo
FROM tracks_raw
LIMIT 5
""")

,track_name,artists,track_genre,popularity,energy,tempo
0,Comedy,Gen Hoshino,acoustic,73,0.4610,87.917
1,Ghost - Acoustic,Ben Woodward,acoustic,55,0.1660,77.489
2,To Begin Again,Ingrid Michaelson;ZAYN,acoustic,57,0.3590,76.332
3,Can't Help Falling In Love,Kina Grannis,acoustic,71,0.0596,181.740
4,Hold On,Chord Overstreet,acoustic,82,0.4430,119.949


## 2. How big is it? `COUNT`

`COUNT(*)` counts rows. `COUNT(DISTINCT track_id)` counts each song only once.
If the two numbers differ, some songs appear more than once. Remember this for section 6.

In [3]:
sql("""
SELECT COUNT(*)                 AS rows,
       COUNT(DISTINCT track_id)  AS different_songs,
       COUNT(DISTINCT track_genre) AS genres
FROM tracks_raw
""")

,rows,different_songs,genres
0,114000,89741,114


Kaggle's page says 125 genres, but the file itself has **114**. Trust the data, not the description.

## 3. Filter rows: `WHERE`, sort them: `ORDER BY`

- `WHERE` keeps only the rows that pass a test.
- `ORDER BY popularity DESC` sorts from highest to lowest (`DESC` = descending, `ASC` = ascending).

In [4]:
sql("""
SELECT COUNT(*) AS songs_90_or_more
FROM tracks_raw
WHERE popularity >= 90
""")

,songs_90_or_more
0,98


In [5]:
sql("""
SELECT track_name, artists, popularity, track_genre
FROM tracks_raw
ORDER BY popularity DESC
LIMIT 5
""")

,track_name,artists,popularity,track_genre
0,Unholy (feat. Kim Petras),Sam Smith;Kim Petras,100,dance
1,Unholy (feat. Kim Petras),Sam Smith;Kim Petras,100,pop
2,"Quevedo: Bzrp Music Sessions, Vol. 52",Bizarrap;Quevedo,99,hip-hop
3,I'm Good (Blue),David Guetta;Bebe Rexha,98,dance
4,I'm Good (Blue),David Guetta;Bebe Rexha,98,edm


Notice *Unholy* appears twice, once as `dance` and once as `pop`. Same song, two rows. We'll come back to this.

## 4. Summarise by group: `GROUP BY`

`GROUP BY track_genre` puts the rows into one pile per genre, then `COUNT` and `AVG` work on each pile separately.
`ROUND(x, 1)` keeps one decimal place.

In [6]:
sql("""
SELECT track_genre,
       COUNT(*)                  AS songs,
       ROUND(AVG(popularity), 1) AS avg_popularity
FROM tracks_raw
GROUP BY track_genre
ORDER BY avg_popularity DESC
LIMIT 10
""")

,track_genre,songs,avg_popularity
0,pop-film,1000,59.3
1,k-pop,1000,56.9
2,chill,1000,53.7
3,sad,1000,52.4
4,grunge,1000,49.6
5,indian,1000,49.5
6,anime,1000,48.8
7,emo,1000,48.1
8,sertanejo,1000,47.9
9,pop,1000,47.6


In [7]:
sql("""
SELECT track_genre,
       ROUND(AVG(popularity), 1) AS avg_popularity
FROM tracks_raw
GROUP BY track_genre
ORDER BY avg_popularity ASC
LIMIT 5
""")

,track_genre,avg_popularity
0,iranian,2.2
1,romance,3.2
2,latin,8.3
3,detroit-techno,11.2
4,chicago-house,12.3


Every genre has exactly 1,000 rows: whoever built the dataset collected 1,000 songs per genre.
The gap between genres is huge (pop-film near 59, iranian near 2), which is why the app asks for the genre.

## 5. Turning units into something readable

`duration_ms` is in milliseconds. Dividing by 60,000 gives minutes.
Writing `60000.0` (with `.0`) makes SQL do decimal division instead of rounding down to whole numbers.

In [8]:
sql("""
SELECT ROUND(AVG(duration_ms) / 60000.0, 2) AS avg_minutes,
       ROUND(MAX(duration_ms) / 60000.0, 1) AS longest_minutes
FROM tracks_raw
""")

,avg_minutes,longest_minutes
0,3.8,87.3


## 6. The duplicate problem

A **subquery** is a query inside another query. The inner one counts copies per song;
the outer one counts how many songs have 1 copy, 2 copies, and so on.

In [9]:
sql("""
SELECT copies, COUNT(*) AS songs
FROM (
    SELECT track_id, COUNT(*) AS copies
    FROM tracks_raw
    GROUP BY track_id
)
GROUP BY copies
ORDER BY copies
""")

,copies,songs
0,1,73100
1,2,11712
2,3,2984
3,4,1372
4,5,431
5,6,117
6,7,22
7,8,2
8,9,1


The record holder: one song filed under 9 genres.

In [10]:
sql("""
SELECT track_name, artists, popularity, track_genre
FROM tracks_raw
WHERE track_id = (
    SELECT track_id FROM tracks_raw
    GROUP BY track_id
    ORDER BY COUNT(*) DESC
    LIMIT 1
)
""")

,track_name,artists,popularity,track_genre
0,Baby Blue - Remastered 2010,Badfinger,67,blues
1,Baby Blue - Remastered 2010,Badfinger,67,country
2,Baby Blue - Remastered 2010,Badfinger,67,folk
3,Baby Blue - Remastered 2010,Badfinger,67,j-pop
4,Baby Blue - Remastered 2010,Badfinger,67,j-rock
5,Baby Blue - Remastered 2010,Badfinger,67,power-pop
6,Baby Blue - Remastered 2010,Badfinger,67,psych-rock
7,Baby Blue - Remastered 2010,Badfinger,67,singer-songwriter
8,Baby Blue - Remastered 2010,Badfinger,67,songwriter


Do the copies at least agree with each other? `HAVING` is like `WHERE`, but it filters **groups** after `GROUP BY`
instead of single rows.

In [11]:
sql("""
SELECT
  (SELECT COUNT(*) FROM (
      SELECT track_id FROM tracks_raw GROUP BY track_id
      HAVING COUNT(*) > 1 AND (MIN(energy) <> MAX(energy) OR MIN(tempo) <> MAX(tempo))
  )) AS songs_whose_copies_differ_in_sound,
  (SELECT COUNT(*) FROM (
      SELECT track_id FROM tracks_raw GROUP BY track_id
      HAVING COUNT(*) > 1 AND MIN(popularity) <> MAX(popularity)
  )) AS songs_whose_copies_differ_in_popularity
""")

,songs_whose_copies_differ_in_sound,songs_whose_copies_differ_in_popularity
0,0,720


The audio measurements are always identical between copies, but 720 songs have copies with *different*
popularity. The dataset was probably collected over several days while the scores moved.

## 7. Other things to decide before cleaning

In [12]:
sql("""
SELECT COUNT(*) AS popularity_zero,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM tracks_raw), 1) AS percent
FROM tracks_raw
WHERE popularity = 0
""")

,popularity_zero,percent
0,16020,14.1


In [13]:
sql("""
SELECT COUNT(*) AS same_name_and_artist_but_different_id
FROM (
    SELECT track_name, artists
    FROM tracks_raw
    GROUP BY track_name, artists
    HAVING COUNT(DISTINCT track_id) > 1
)
""")

,same_name_and_artist_but_different_id
0,4657


In [14]:
sql("""
SELECT COUNT(*) AS rows_missing_name_or_artist
FROM tracks_raw
WHERE track_name IS NULL OR artists IS NULL
""")

,rows_missing_name_or_artist
0,1


- **16,020 songs (14%) have popularity 0.** Are they really unpopular, or is 0 Spotify's "no data" value?
- **4,657 songs share a name and artist but have different IDs**, e.g. the single and the album version of the same song.
- **1 row** has no song name or artist.

These are decisions for the cleaning step, and they're yours to make.

## Your turn

Write these two yourself. Use the sections above as a pattern.

**Exercise 1.** How many songs are marked `explicit`? (`explicit` is stored as 1 for yes, 0 for no.)

In [17]:
# Exercise 1: write your query on the empty line, then remove the # from all three lines
sql("""
select count(*) as explicit_songs
from tracks_raw
where explicit = 1
 """)

,explicit_songs
0,9747


**Exercise 2.** Do explicit songs have a higher average popularity than non-explicit ones? (Hint: `GROUP BY explicit`.)

In [19]:
# Exercise 2: same as above
sql("""
SELECT explicit,
       COUNT(*)                  AS songs,
       ROUND(AVG(popularity), 1) AS avg_popularity
FROM tracks_raw
GROUP BY explicit
""")

,explicit,songs,avg_popularity
0,0,104253,32.9
1,1,9747,36.5
